In [14]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns
import scipy
import sklearn
from sklearn.svm import SVC
from scipy.stats import pearsonr
from sklearn import datasets, linear_model
from sklearn import preprocessing
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import (
    train_test_split, 
    StratifiedKFold, cross_val_score,  
    RepeatedStratifiedKFold, 
    RandomizedSearchCV,
    train_test_split, 
    KFold
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
    f1_score
)
from sklearn.feature_selection import SelectFromModel
from sklearn.compose import ColumnTransformer
from scipy.stats import loguniform
from scipy import sparse
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import make_scorer, balanced_accuracy_score, f1_score

#!pip install xlrd 
#!pip install category_encoders
import category_encoders as ce



In [2]:
df = pd.read_excel("TrainDataset2025.xls")
df.head()

,ID,pCR (outcome),RelapseFreeSurvival (outcome),Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,TRG002174,1,144.0,41.0,0,0,0,1,3,3,...,0.517172,0.375126,3.325332,0.002314,3880771.500,473.464852,0.000768,0.182615,0.030508,0.000758
1,TRG002178,0,142.0,39.0,1,1,0,0,3,3,...,0.444391,0.444391,3.032144,0.005612,2372009.744,59.459710,0.004383,0.032012,0.001006,0.003685
2,TRG002204,1,135.0,31.0,0,0,0,1,2,1,...,0.534549,0.534549,2.485848,0.006752,1540027.421,33.935384,0.007584,0.024062,0.000529,0.006447
3,TRG002206,0,12.0,35.0,0,0,0,1,3,3,...,0.506185,0.506185,2.606255,0.003755,6936740.794,46.859265,0.005424,0.013707,0.000178,0.004543
4,TRG002210,0,109.0,61.0,1,0,0,0,2,1,...,0.462282,0.462282,2.809279,0.006521,1265399.054,39.621023,0.006585,0.034148,0.001083,0.005626


In [3]:
df.info()
df.describe()
df.replace(999, pd.NA, inplace=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Columns: 121 entries, ID to original_ngtdm_Strength
dtypes: float64(108), int64(12), object(1)
memory usage: 378.3+ KB


In [4]:
missing_cols = df.isna().sum()
missing_cols = missing_cols[missing_cols > 0]
# Get columns with missing values
missing_cols = df.columns[df.isna().any()]

# Missing data summary
summary = pd.DataFrame({
    'Column': missing_cols,
    'MissingCount': [df[col].isna().sum() for col in missing_cols],
    'Dtype': [df[col].dtype for col in missing_cols]
})

print(summary)

            Column  MissingCount   Dtype
0    pCR (outcome)             5  object
1              PgR             1  object
2             HER2             1  object
3  TrippleNegative             1  object
4       ChemoGrade             3  object
5    Proliferation             2  object
6    HistologyType             3  object
7         LNStatus             1  object
8             Gene            88  object


In [5]:
df_dropped =  df.dropna(subset=['pCR (outcome)'])
df_dropped = df_dropped.drop("RelapseFreeSurvival (outcome)", axis=1)
df_dropped.shape

(395, 120)

In [6]:
missing_cols = df_dropped.columns[df_dropped.isna().any()]
# Missing data summary
summary = pd.DataFrame({
    'Column': missing_cols,
    'MissingCount': [df_dropped[col].isna().sum() for col in missing_cols],
    'Dtype': [df_dropped[col].dtype for col in missing_cols]
})

print(summary)

            Column  MissingCount   Dtype
0              PgR             1  object
1             HER2             1  object
2  TrippleNegative             1  object
3       ChemoGrade             3  object
4    Proliferation             2  object
5    HistologyType             3  object
6         LNStatus             1  object
7             Gene            85  object


In [23]:
# Iterative imputation

df_imputed = df_dropped.copy()

## Select categorical columns
cat_columns = df_imputed.select_dtypes(include=['object', 'category']).columns
cat_cols_to_encode = cat_columns.drop('pCR (outcome)')

encoder = ce.OrdinalEncoder(handle_missing='return_nan') 


## Encode categorical features
df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])


## Imputation with IterativeImputer
y_target = df_imputed['pCR (outcome)']
X_features = df_imputed.drop(columns=['pCR (outcome)'])

# Iterative Imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Convert back to DataFrame
X_imputed_df = pd.DataFrame(X_imputed,
                            columns=X_features.columns,
                            index=X_features.index)

# Round encoded categorical columns back to integers
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

    
# Inverse transform encoded categorical columns
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])


# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]
print("\n--- Missing Value Check ---")
print(final_df.isna().sum().sum())



y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)", "ID"], axis=1)

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


# Train / test split (hold out 30% for final evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

keep_cats = ["Gene", "HER2", "ER"] 
other_cats = [c for c in cat_features if c not in keep_cats]

keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop"
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop"
)




--- Missing Value Check ---
0


In [8]:
# Feature selectors
rf_selector = SelectFromModel(
    RandomForestClassifier(
        n_estimators=500,
        max_depth=20,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    ),
    threshold="median", # mean better result using median
)

from xgboost import XGBClassifier
from sklearn.feature_selection import SelectFromModel

xgb_selector = SelectFromModel( # poor result
    XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1.5,  # imbalance handling
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss"
    ),
    threshold="median"
)

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

logreg_l1_selector = SelectFromModel( # poor result
    LogisticRegression(
        penalty="l1",
        solver="saga",
        class_weight="balanced",
        random_state=42,
        max_iter=5000,
    ),
    threshold="median"  # or "mean", or "0.5*median"
)


from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier

boruta_selector = BorutaPy( # slow and poor result
    estimator=RandomForestClassifier(
        n_estimators=1000, 
        class_weight="balanced", 
        random_state=42,
        n_jobs=-1),
    n_estimators="auto",
    max_iter=50
)

from sklearn.feature_selection import RFE
from sklearn.svm import LinearSVC

rfe_selector = RFE(
    estimator=LinearSVC(
        penalty="l2",
        class_weight="balanced",
        random_state=42,
    ),
    n_features_to_select=42,   # tune based on performance
    step=0.1
)

In [9]:
# SVM 
## Features = [ER/HER2/Gene one-hot] + [RF-selected other features]
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector), 
    ])),
])

svm_pipe = Pipeline(steps=[
    ("features", full_features),
    ("svm", SVC(kernel="rbf", probability=False)),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)


param_dist = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__class_weight": [None, "balanced"],
}

search = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)


search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC AUC:", search.best_score_)

best_svm = search.best_estimator_

# Evaluation of Model
# Fit best pipeline on full training set
best_svm.fit(X_train, y_train)

# Decision scores on test set
scores_test = best_svm.decision_function(X_test)  # continuous margins
y_pred_default = best_svm.predict(X_test)
# y_pred_best = (scores_test >= thr_bal).astype(int)

# Metrics with default decision threshold
roc_auc = roc_auc_score(y_test, scores_test)
pr_auc = average_precision_score(y_test, scores_test)  # PR AUC
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix (default threshold):\n", confusion_matrix(y_test, y_pred_default))
print("Classification report (default threshold):\n", classification_report(y_test, y_pred_default))

Fitting 25 folds for each of 50 candidates, totalling 1250 fits
Best params: {'svm__C': np.float64(361.2478500429091), 'svm__class_weight': 'balanced', 'svm__gamma': np.float64(0.00037961668958008145)}
Best CV ROC AUC: 0.6903497981933501
Test ROC AUC: 0.7638297872340426
Test PR AUC: 0.5205332717241044
Test balanced accuracy: 0.7257446808510639
Confusion matrix (default threshold):
 [[65 29]
 [ 6 19]]
Classification report (default threshold):
               precision    recall  f1-score   support

           0       0.92      0.69      0.79        94
           1       0.40      0.76      0.52        25

    accuracy                           0.71       119
   macro avg       0.66      0.73      0.65       119
weighted avg       0.81      0.71      0.73       119



In [33]:
# Random Forest
bal_acc_scorer = make_scorer(balanced_accuracy_score)
f1_pos_scorer = make_scorer(f1_score, pos_label=1)
# Combine "always keep" block with RF-selected block
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        #("select", rf_selector),
        ("select", boruta_selector),
    ])),
])

# -----------------------------
# 5. Random Forest classifier + tuning
# -----------------------------
rf_clf = RandomForestClassifier(random_state=42)

rf_pipe = Pipeline(steps=[
    ("features", full_features),
    ("rf", rf_clf),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

# Hyperparameter search space (advanced RF)
param_dist_rf = {
    "rf__n_estimators": [100, 200, 300, 500, 600],
    "rf__max_depth": [None, 3, 5, 7, 9, 11, 15],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4],
    "rf__max_features": ["sqrt", "log2", 0.3, 0.5, None],
    "rf__bootstrap": [True, False],
    "rf__class_weight": [None, "balanced", "balanced_subsample"],
}

search_rf = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=param_dist_rf,
    n_iter=60,               # adjust if too slow
    cv=cv,
    scoring=bal_acc_scorer, #"roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

print("\nFitting RandomizedSearchCV for RandomForest...")
search_rf.fit(X_train, y_train)

print("\nBest RF params:", search_rf.best_params_)
print("Best CV ROC AUC:", search_rf.best_score_)

best_rf = search_rf.best_estimator_

# -----------------------------
# 6. Final evaluation on test set
# -----------------------------
best_rf.fit(X_train, y_train)

# For RF we use predict_proba for continuous scores
proba_test = best_rf.predict_proba(X_test)[:, 1]
y_pred_default = best_rf.predict(X_test)  # default threshold 0.5

roc_auc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("\n--- Random Forest Test Performance (default threshold 0.5) ---")
print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_default))
print("Classification report:\n", classification_report(y_test, y_pred_default))



Fitting RandomizedSearchCV for RandomForest...
Fitting 25 folds for each of 60 candidates, totalling 1500 fits

Best RF params: {'rf__n_estimators': 300, 'rf__min_samples_split': 2, 'rf__min_samples_leaf': 4, 'rf__max_features': 0.5, 'rf__max_depth': 11, 'rf__class_weight': 'balanced_subsample', 'rf__bootstrap': False}
Best CV ROC AUC: 0.6574982381959126

--- Random Forest Test Performance (default threshold 0.5) ---
Test ROC AUC: 0.6772340425531915
Test PR AUC: 0.38994681062753767
Test balanced accuracy: 0.597872340425532
Confusion matrix:
 [[56 38]
 [10 15]]
Classification report:
               precision    recall  f1-score   support

           0       0.85      0.60      0.70        94
           1       0.28      0.60      0.38        25

    accuracy                           0.60       119
   macro avg       0.57      0.60      0.54       119
weighted avg       0.73      0.60      0.63       119



In [28]:
# ANN
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector),
    ])),
])

# -----------------------------
# 5. Fit preprocessing + feature selection
# -----------------------------
print("\nFitting preprocessing + RF feature selection...")
full_features.fit(X_train, y_train)

X_train_proc = full_features.transform(X_train)
X_test_proc = full_features.transform(X_test)

# Convert sparse to dense for Keras
if sparse.issparse(X_train_proc):
    X_train_proc = X_train_proc.toarray()
if sparse.issparse(X_test_proc):
    X_test_proc = X_test_proc.toarray()

print("Processed train shape:", X_train_proc.shape)
print("Processed test shape:", X_test_proc.shape)

# -----------------------------
# 6. Build ANN model in TensorFlow / Keras
# -----------------------------
input_dim = X_train_proc.shape[1]

from tensorflow.keras import layers, regularizers

def build_ann_model(input_dim: int) -> keras.Model:
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        
        # Block 1
        layers.Dense(
            128,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4),
        ),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # Block 2
        layers.Dense(
            128,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4),
        ),
        layers.BatchNormalization(),
        layers.Dropout(0.4),

        # Block 3
        layers.Dense(
            64,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4),
        ),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        # Block 3
        layers.Dense(
            64,
            activation="relu",
            kernel_regularizer=regularizers.l2(1e-4),
        ),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        
        # Output
        layers.Dense(1, activation="sigmoid"),  # binary classification
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.AUC(name="auc"),
            keras.metrics.AUC(name="pr_auc", curve="PR"),
            keras.metrics.BinaryAccuracy(name="accuracy"),
        ],
    )
    return model

model = build_ann_model(input_dim)

# -----------------------------
# 7. Class weights + callbacks
# -----------------------------
classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)
class_weight_dict = {cls: w for cls, w in zip(classes, class_weights_array)}
print("\nClass weights:", class_weight_dict)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=15,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1,
)

# -----------------------------
# 8. Train ANN (with validation split)
# -----------------------------
history = model.fit(
    X_train_proc,
    y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weight_dict,
    verbose=1,
)

# -----------------------------
# 9. Evaluation on test set
# -----------------------------
# Probabilities (for class 1)
proba_test = model.predict(X_test_proc).ravel()
y_pred_default = (proba_test >= 0.5).astype(int)  # default threshold 0.5

roc_auc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("\n--- ANN Test Performance (default threshold 0.5) ---")
print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_default))
print("Classification report:\n", classification_report(y_test, y_pred_default))


Fitting preprocessing + RF feature selection...
Processed train shape: (276, 68)
Processed test shape: (119, 68)

Class weights: {np.int64(0): np.float64(0.6359447004608295), np.int64(1): np.float64(2.3389830508474576)}
Epoch 1/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - accuracy: 0.4955 - auc: 0.4995 - loss: 1.0241 - pr_auc: 0.2191 - val_accuracy: 0.6964 - val_auc: 0.5781 - val_loss: 0.6732 - val_pr_auc: 0.3329 - learning_rate: 0.0010
Epoch 2/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5500 - auc: 0.5494 - loss: 0.9147 - pr_auc: 0.2883 - val_accuracy: 0.7321 - val_auc: 0.6797 - val_loss: 0.6817 - val_pr_auc: 0.4460 - learning_rate: 0.0010
Epoch 3/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4773 - auc: 0.4499 - loss: 1.0195 - pr_auc: 0.1772 - val_accuracy: 0.6786 - val_auc: 0.7117 - val_loss: 0.6949 - val_pr_auc: 0.4392 - learning_rate: 0.0010
Epoch 4/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5364 - auc: 0.5739 - loss: 0.8259 - pr_auc: 0.2397 - va

In [32]:
# this is the SVM that we will use. 
# SVM with threshold tuningf
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector),
    ])),
])

# =========================================================
# 2. Inner train/validation split for threshold tuning
# =========================================================
X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42,
)

# =========================================================
# 3. SVM pipeline + hyperparameter tuning on inner train
# =========================================================
svm_pipe = Pipeline(steps=[
    ("features", full_features),
    ("svm", SVC(kernel="rbf", probability=False)),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

param_dist_svm = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__class_weight": [None, "balanced"],
}

bal_acc_scorer = make_scorer(balanced_accuracy_score)
f1_pos_scorer = make_scorer(f1_score, pos_label=1)

search_svm = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=param_dist_svm,
    n_iter=50,
    cv=cv,
    scoring=bal_acc_scorer, #f1_pos_scorer, #"roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

print("\nFitting RandomizedSearchCV for SVM on inner training data...")
search_svm.fit(X_train_inner, y_train_inner)

print("\nBest SVM params:", search_svm.best_params_)
print("Best inner CV ROC AUC:", search_svm.best_score_)

best_svm_inner = search_svm.best_estimator_

# =========================================================
# 4. Threshold tuning on validation set (Youden / balanced acc)
# =========================================================
def tune_threshold_from_scores(scores, y_true):
    """
    Find the threshold that maximises balanced accuracy using
    the ROC curve (i.e., Youden's J = TPR - FPR).
    This uses all distinct score values as candidate thresholds,
    so it's exact given the scores.
    """
    scores = np.asarray(scores)
    y_true = np.asarray(y_true)

    # fpr, tpr, thresholds: thresholds are sorted from high to low
    fpr, tpr, thresholds = roc_curve(y_true, scores)

    # balanced accuracy = (TPR + TNR) / 2 = (TPR + (1 - FPR)) / 2
    bal_acc = (tpr + (1 - fpr)) / 2.0

    # index of maximum balanced accuracy
    idx = np.argmax(bal_acc)
    best_thr = thresholds[idx]
    best_bal = bal_acc[idx]

    return best_thr, best_bal
    
from sklearn.metrics import roc_curve, f1_score

def tune_threshold_for_f1(scores, y_true, positive_label=1):
    """
    Find the threshold that maximises F1-score for the positive class.
    Thresholds are taken from unique score cutpoints returned by roc_curve,
    so this is exact (no arbitrary linear spacing).
    """
    scores = np.asarray(scores)
    y_true = np.asarray(y_true)

    # Generate all possible score cutpoints that change classification
    fpr, tpr, thresholds = roc_curve(y_true, scores)

    best_thr = thresholds[0]
    best_f1 = -1.0

    # Iterate over all meaningful threshold candidates
    for thr in thresholds:
        y_pred = (scores >= thr).astype(int)
        f1 = f1_score(y_true, y_pred, pos_label=positive_label)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    return best_thr, best_f1


    
# decision_function gives continuous margins (good for threshold tuning)
scores_val = best_svm_inner.decision_function(X_val)
best_thr, best_bal_val = tune_threshold_from_scores(scores_val, y_val)

print(f"\nBest SVM threshold on validation (by balanced accuracy): {best_thr:.4f}")
print(f"Validation balanced accuracy at best threshold: {best_bal_val:.4f}")

y_val_pred_tuned = (scores_val >= best_thr).astype(int)

print("\n--- Validation metrics at tuned threshold ---")
print("ROC AUC (val):", roc_auc_score(y_val, scores_val))
print("PR AUC (val):", average_precision_score(y_val, scores_val))
print("Balanced accuracy (val):", balanced_accuracy_score(y_val, y_val_pred_tuned))
print("Confusion matrix (val):\n", confusion_matrix(y_val, y_val_pred_tuned))
print("Classification report (val):\n", classification_report(y_val, y_val_pred_tuned))

# =========================================================
# 5. Refit SVM on full training data (inner + val) and evaluate on test
# =========================================================
best_svm_final = search_svm.best_estimator_
best_svm_final.fit(X_train, y_train)   # X_train = inner + val

scores_test = best_svm_final.decision_function(X_test)

# Default 0-threshold (what .predict() uses)
y_test_pred_default = best_svm_final.predict(X_test)

# Tuned threshold from validation
y_test_pred_tuned = (scores_test >= best_thr).astype(int)

print("\n=== SVM Test Performance (default decision threshold) ===")
print("Test ROC AUC:", roc_auc_score(y_test, scores_test))
print("Test PR AUC:", average_precision_score(y_test, scores_test))
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred_default))
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred_default))
print("Classification report:\n", classification_report(y_test, y_test_pred_default))

print("\n=== SVM Test Performance (tuned threshold from validation) ===")
print("Threshold used:", best_thr)
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred_tuned))
print("Confusion matrix (tuned):\n", confusion_matrix(y_test, y_test_pred_tuned))
print("Classification report (tuned):\n", classification_report(y_test, y_test_pred_tuned))



Fitting RandomizedSearchCV for SVM on inner training data...
Fitting 25 folds for each of 50 candidates, totalling 1250 fits

Best SVM params: {'svm__C': np.float64(43.00001586162607), 'svm__class_weight': 'balanced', 'svm__gamma': np.float64(0.002933870049164396)}
Best inner CV ROC AUC: 0.41676550502637455

Best SVM threshold on validation (by balanced accuracy): 0.1655
Validation balanced accuracy at best threshold: 0.6402

--- Validation metrics at tuned threshold ---
ROC AUC (val): 0.6590909090909092
PR AUC (val): 0.32986341787662565
Balanced accuracy (val): 0.6401515151515151
Confusion matrix (val):
 [[27 17]
 [ 4  8]]
Classification report (val):
               precision    recall  f1-score   support

           0       0.87      0.61      0.72        44
           1       0.32      0.67      0.43        12

    accuracy                           0.62        56
   macro avg       0.60      0.64      0.58        56
weighted avg       0.75      0.62      0.66        56


=== SVM T

In [37]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import (
    make_scorer,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    f1_score,
)
import numpy as np

# =========================================================
# 0. Helper: scorer + F1-based threshold tuning
# =========================================================
bal_acc_scorer = make_scorer(balanced_accuracy_score)
# or, if you prefer to target F1 for class 1 in CV, use:
f1_pos_scorer = make_scorer(f1_score, pos_label=1)

def tune_threshold_for_f1(scores, y_true, positive_label=1):
    """
    Find threshold that maximises F1 for the positive class.
    Thresholds from roc_curve => exact search.
    """
    scores = np.asarray(scores)
    y_true = np.asarray(y_true)

    fpr, tpr, thresholds = roc_curve(y_true, scores, pos_label=positive_label)

    best_thr = thresholds[0]
    best_f1 = -1.0

    for thr in thresholds:
        y_pred = (scores >= thr).astype(int)
        f1 = f1_score(y_true, y_pred, pos_label=positive_label)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    return best_thr, best_f1

# =========================================================
# 1. Inner train/validation split (for threshold tuning)
# =========================================================
X_train_inner, X_val_xgb, y_train_inner, y_val_xgb = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42,
)

# =========================================================
# 2. Base XGBoost model + pipeline
# =========================================================
# Class imbalance weight based on training data
pos_weight = (y_train_inner == 0).sum() / (y_train_inner == 1).sum()

xgb_clf = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric="logloss",
)

xgb_pipe = Pipeline(steps=[
    ("features", full_features),  # your FeatureUnion with preprocessing + selector
    ("xgb", xgb_clf),
])

# CV setup (you can reuse the same cv object as SVM/RF)
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

param_dist_xgb = {
    "xgb__n_estimators": [200, 400, 600],
    "xgb__max_depth": [3, 4, 5],
    "xgb__learning_rate": [0.01, 0.05, 0.1],
    "xgb__subsample": [0.7, 0.8, 1.0],
    "xgb__colsample_bytree": [0.7, 0.8, 1.0],
}

search_xgb = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=param_dist_xgb,
    n_iter=30,
    cv=cv,
    scoring="roc_auc", #f1_pos_scorer, bal_acc_scorer,  # or f1_pos_scorer if you prefer
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

print("\nFitting RandomizedSearchCV for XGBoost on inner training data...")
search_xgb.fit(X_train_inner, y_train_inner)

print("\nBest XGB params:", search_xgb.best_params_)
print("Best inner CV balanced accuracy:", search_xgb.best_score_)

best_xgb_inner = search_xgb.best_estimator_

# =========================================================
# 3. Threshold tuning on validation set (F1 for class 1)
# =========================================================
proba_val_xgb = best_xgb_inner.predict_proba(X_val_xgb)[:, 1]

best_thr_xgb, best_f1_val_xgb = tune_threshold_for_f1(
    proba_val_xgb,
    y_val_xgb,
    positive_label=1,
)

print(f"\n[XGB] Best threshold on validation (F1 for class 1): {best_thr_xgb:.4f}")
print(f"[XGB] Validation F1(1) at best threshold: {best_f1_val_xgb:.4f}")

y_val_pred_xgb_tuned = (proba_val_xgb >= best_thr_xgb).astype(int)

print("\n--- XGB validation metrics at tuned threshold ---")
print("ROC AUC (val):", roc_auc_score(y_val_xgb, proba_val_xgb))
print("PR AUC (val):", average_precision_score(y_val_xgb, proba_val_xgb))
print("Balanced accuracy (val):", balanced_accuracy_score(y_val_xgb, y_val_pred_xgb_tuned))
print("Confusion matrix (val):\n", confusion_matrix(y_val_xgb, y_val_pred_xgb_tuned))
print("Classification report (val):\n", classification_report(y_val_xgb, y_val_pred_xgb_tuned))

# =========================================================
# 4. Refit XGB on full training data (inner + val) and evaluate on test
# =========================================================
best_xgb_final = search_xgb.best_estimator_
best_xgb_final.fit(X_train, y_train)  # X_train = inner + val

proba_test_xgb = best_xgb_final.predict_proba(X_test)[:, 1]

# Default threshold 0.5
y_test_pred_xgb_default = (proba_test_xgb >= 0.5).astype(int)

# Tuned threshold from validation
y_test_pred_xgb_tuned = (proba_test_xgb >= best_thr_xgb).astype(int)

print("\n=== XGB Test Performance (default threshold 0.5) ===")
print("Test ROC AUC:", roc_auc_score(y_test, proba_test_xgb))
print("Test PR AUC:", average_precision_score(y_test, proba_test_xgb))
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred_xgb_default))
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred_xgb_default))
print("Classification report:\n", classification_report(y_test, y_test_pred_xgb_default))

print("\n=== XGB Test Performance (tuned threshold from validation) ===")
print("Threshold used:", best_thr_xgb)
print("Test balanced accuracy (tuned):", balanced_accuracy_score(y_test, y_test_pred_xgb_tuned))
print("Confusion matrix (tuned):\n", confusion_matrix(y_test, y_test_pred_xgb_tuned))
print("Classification report (tuned):\n", classification_report(y_test, y_test_pred_xgb_tuned))



Fitting RandomizedSearchCV for XGBoost on inner training data...
Fitting 25 folds for each of 30 candidates, totalling 750 fits

Best XGB params: {'xgb__subsample': 0.7, 'xgb__n_estimators': 400, 'xgb__max_depth': 3, 'xgb__learning_rate': 0.05, 'xgb__colsample_bytree': 0.7}
Best inner CV balanced accuracy: 0.7368039215686275

[XGB] Best threshold on validation (F1 for class 1): 0.7293
[XGB] Validation F1(1) at best threshold: 0.4800

--- XGB validation metrics at tuned threshold ---
ROC AUC (val): 0.7528409090909091
PR AUC (val): 0.40118502770941794
Balanced accuracy (val): 0.6704545454545454
Confusion matrix (val):
 [[37  7]
 [ 6  6]]
Classification report (val):
               precision    recall  f1-score   support

           0       0.86      0.84      0.85        44
           1       0.46      0.50      0.48        12

    accuracy                           0.77        56
   macro avg       0.66      0.67      0.67        56
weighted avg       0.77      0.77      0.77        56